# Cross-Cutting: Logging, Secret Scrubbing, Typed Errors

Covers task backlog §3.4. `@logged_tool` (in
[tool_logging.py](tool_logging.py)) wraps both tools -- applied via Python decorator
stacking, not the MCP SDK's low-level `ServerMiddleware` protocol (that hooks every
inbound request/notification at the wire-protocol level, with its own deadlock caveats
around server-to-client calls; wrapping just the two tool functions is simpler and
fully within this project's control for what's actually needed here).

**Real risk worth stating up front:** stacking `@logged_tool` under `@mcp_server.tool`
could silently have broken JSON schema generation (if schema introspection didn't see
through the wrapper). Checked directly, not assumed -- `functools.wraps` sets
`__wrapped__`, and `inspect.signature()` follows it by default, so the schema
generation still sees the original function's real signature. Verified below, not
just claimed.

In [1]:
# !pip install mcp tavily-python

## Decorator stacking didn't break schema generation

In [2]:
from mcp_server import mcp_server

tools = await mcp_server.list_tools()
for t in tools:
    print(t.name, "->", list(t.input_schema["properties"]))

rag_schema = next(t.input_schema for t in tools if t.name == "rag.search")
assert set(rag_schema["properties"]) == {"query", "top_k", "budget_min", "budget_max", "category", "brand"}
web_schema = next(t.input_schema for t in tools if t.name == "web.search")
assert set(web_schema["properties"]) == {"query", "top_k"}
print("\n[OK] both schemas still correct through the @logged_tool wrapper.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

rag.search -> ['query', 'top_k', 'budget_min', 'budget_max', 'category', 'brand']
web.search -> ['query', 'top_k']

[OK] both schemas still correct through the @logged_tool wrapper.


## Structured logging: request, response, timestamp, source URLs

One real `rag.search` call (free -- local embedding model + Chroma, no external API) to
check the log entry is real and complete, not a placeholder.

In [3]:
import time

import tool_logging
from mcp_server import rag_search

before = len(tool_logging.TOOL_CALL_LOG)
result = rag_search(query="insulated lunch box for kids", top_k=2)
entry = tool_logging.TOOL_CALL_LOG[-1]

print("tool:", entry["tool"])
print("request:", entry["request"])
print("source_urls:", entry["source_urls"])
print("error_type:", entry["error_type"])

assert len(tool_logging.TOOL_CALL_LOG) == before + 1, "exactly one entry should have been logged"
assert entry["tool"] == "rag.search"
assert entry["request"] == {"query": "insulated lunch box for kids", "top_k": 2}
assert abs(entry["timestamp"] - time.time()) < 10, "timestamp should be real and recent"
assert len(entry["source_urls"]) == len(result["products"]) > 0, "every product's url should be captured"
assert entry["error_type"] is None
print("\n[OK] request/response/timestamp/source_urls all captured correctly for a real call.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tool: rag.search
request: {'query': 'insulated lunch box for kids', 'top_k': 2}
source_urls: ['https://www.amazon.com/Wildkin-Insulated-Moisture-Resistant-Organization/dp/B004NWPOYI', 'https://www.amazon.com/Wildkin-Insulated-Moisture-Resistant-Organization/dp/B004NWLYT2']
error_type: None

[OK] request/response/timestamp/source_urls all captured correctly for a real call.


## Secret scrubbing

Two layers: by-value (redacts whatever the currently-configured key actually is) and
by-shape (redacts anything that merely *looks* like an API key, even one never
configured here) -- so a secret still gets caught if the log ever picks one up from an
unexpected source (e.g. an underlying HTTP client dumping a request header into an
error message).

In [4]:
import os

from tool_logging import scrub_secrets

real_key = os.environ["TAVILY_API_KEY"]
leaked = f"request failed, key was {real_key}, please retry"
scrubbed = scrub_secrets(leaked)
print("by-value:", scrubbed)
assert real_key not in scrubbed and "***REDACTED***" in scrubbed

shape_leaked = "leaked sk-abcdefghijklmnopqrstuvwxyz123456 in a stray log line"
shape_scrubbed = scrub_secrets(shape_leaked)
print("by-shape:", shape_scrubbed)
assert "sk-abcdefgh" not in shape_scrubbed and "***REDACTED***" in shape_scrubbed

# Recursive scrubbing -- nested inside a dict/list, the way a real request/response is shaped
from tool_logging import _scrub_recursive

nested = {"error": f"auth failed: {real_key}", "context": ["ok", f"also has {real_key} here"]}
scrubbed_nested = _scrub_recursive(nested)
print("nested:", scrubbed_nested)
assert real_key not in str(scrubbed_nested)

print("\n[OK] secrets scrubbed by value, by shape, and recursively through nested structures.")

by-value: request failed, key was ***REDACTED***, please retry
by-shape: leaked ***REDACTED*** in a stray log line
nested: {'error': 'auth failed: ***REDACTED***', 'context': ['ok', 'also has ***REDACTED*** here']}

[OK] secrets scrubbed by value, by shape, and recursively through nested structures.


## Typed error contract

Mocked (same reasoning as §3.3: can't reliably force a real timeout on demand). Checks
that a known failure mode surfaces as a `ToolError` with a specific, branchable
`error_type` -- not just free text -- both to the caller and in the log.

In [5]:
from unittest.mock import MagicMock, patch

import web_search_tool as wst
from mcp_server import web_search
from tool_logging import ToolError

wst._cache.clear()
always_fails = MagicMock()
always_fails.search.side_effect = wst.TavilyTimeoutError(1.0)

with patch.object(wst, "_get_client", return_value=always_fails), patch.object(wst.time, "sleep"):
    try:
        web_search(query="unique typed-error test", top_k=1)
        caught = None
    except ToolError as e:
        caught = e

assert caught is not None and caught.error_type == "timeout"
print("[OK] caller sees ToolError(error_type='timeout'), not a free-text-only exception")

log_entry = tool_logging.TOOL_CALL_LOG[-1]
assert log_entry["error_type"] == "timeout" and log_entry["response"] is None
print("[OK] the log entry also captured the typed error_type, not just that *something* failed")

# An unrelated exception type (not one of the known Tavily ones) still gets typed, just generically
wst._cache.clear()
weird_failure = MagicMock()
weird_failure.search.side_effect = ValueError("something totally unexpected")
with patch.object(wst, "_get_client", return_value=weird_failure):
    try:
        web_search(query="unique unexpected-error test", top_k=1)
        caught2 = None
    except ToolError as e:
        caught2 = e

assert caught2 is not None and caught2.error_type == "internal_error"
print("[OK] an unanticipated exception still comes out typed (falls back to 'internal_error'), not unhandled")

[OK] caller sees ToolError(error_type='timeout'), not a free-text-only exception
[OK] the log entry also captured the typed error_type, not just that *something* failed
[OK] an unanticipated exception still comes out typed (falls back to 'internal_error'), not unhandled


## Next steps

- Write the MCP README (this section's last remaining backlog item)
- The Retriever swap (§2.4's open item) + persistent server/session lifecycle + conflict
  reconciliation -- planned as one consolidated piece of work, not part of §3